# 1. Experiment Overview

This notebook implements the experimental pipeline used to compare tree-based machine learning algorithms for binary classification under different dataset characteristics.

The experiments analyze the impact of:

- dataset size
- class imbalance

The following algorithms are evaluated:

- Decision Tree (CART)
- Random Forest
- Extra Trees
- Gradient Boosting
- XGBoost
- LightGBM
- CatBoost

The pipeline performs the following steps:

1. dataset loading and preprocessing  
2. dataset size and class imbalance generation  
3. hyperparameter optimization  
4. model training and evaluation  
5. statistical significance testing  
6. result aggregation and visualization

## 2. Project Structure and Execution

This notebook runs the experimental pipeline for the binary classification part of the study and interacts with the GitHub repository that stores datasets, tuned hyperparameters, and previously generated results.

When executed in Google Colab, the repository is cloned so that the notebook can access the required datasets and configuration files.

The repository contains:

- `datasets/original_datasets`  
  Original datasets used in the experiments.

- `datasets/generated_datasets`  
  Generated dataset subsets for dataset size and class imbalance experiments.

- `config`  
  Tuned hyperparameters stored as JSON files.

- `results`  
  Experimental results generated by the notebook.

- `results/figures`  
  Visualizations produced from experiment results.

During execution, the notebook generates intermediate datasets, results, and figures in the runtime environment. These files can then be downloaded and uploaded to the repository if needed.

In [1]:
import os

REPO_NAME = "Tree-algorithms-dataset-characteristics"
REPO_URL = "https://github.com/Ilaha-Habibova/Tree-algorithms-dataset-characteristics.git"

if "COLAB_GPU" in os.environ and not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}

if "COLAB_GPU" in os.environ and os.path.exists(REPO_NAME):
    %cd {REPO_NAME}

BASE_PATH = os.getcwd()

ORIGINAL_DATASETS_PATH = os.path.join(BASE_PATH, "datasets", "original_datasets")
CONFIG_PATH = os.path.join(BASE_PATH, "config")
RESULTS_PATH = os.path.join(BASE_PATH, "results")

os.makedirs(CONFIG_PATH, exist_ok=True)
os.makedirs(RESULTS_PATH, exist_ok=True)

print("Repository ready.")

Cloning into 'Tree-algorithms-dataset-characteristics'...
remote: Enumerating objects: 447, done.
remote: Counting objects: 100% (122/122), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 447 (delta 87), reused 45 (delta 45), pack-reused 325 (from 2)
Receiving objects: 100% (447/447), 20.33 MiB | 8.76 MiB/s, done.
Resolving deltas: 100% (181/181), done.
/content/Tree-algorithms-dataset-characteristics
Repository ready.


## 3. Environment Setup and Library Versions

This section installs the required libraries and imports the packages used throughout the experimental pipeline.

The libraries support:

- data manipulation and numerical computation
- implementation of machine learning algorithms
- statistical significance testing
- visualization of experimental results

For reproducibility, the versions of the main libraries used in the experiment are also recorded.

In [2]:
# Install required libraries
!pip install -q catboost scikit-posthocs itables requests

# ALL IMPORTS
import os
import json
import time
import random
import warnings
import pickle
import requests
import shutil

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
import scikit_posthocs as sp

from sklearn.model_selection import (
    StratifiedKFold,
    StratifiedShuffleSplit,
    RandomizedSearchCV,
    cross_validate
)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.base import clone

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier
)

import xgboost as xgb
import lightgbm as lgb
import catboost as cb

warnings.filterwarnings("ignore")

# Random Seed Configuration
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

print("Random seed set to:", SEED)

# GLOBAL CV
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

# Model Size Utility
def get_model_size_kb(model):
    return len(pickle.dumps(model)) / 1024

# Library versions (reproducibility)
import sklearn
import scipy
import matplotlib
import scikit_posthocs

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("xgboost:", xgb.__version__)
print("lightgbm:", lgb.__version__)
print("catboost:", cb.__version__)
print("scipy:", scipy.__version__)
print("matplotlib:", matplotlib.__version__)
print("seaborn:", sns.__version__)
print("scikit-posthocs:", scikit_posthocs.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 34.4 MB/s eta 0:00:00
Random seed set to: 42
numpy: 2.0.2
pandas: 2.2.2
scikit-learn: 1.6.1
xgboost: 3.2.0
lightgbm: 4.6.0
catboost: 1.2.10
scipy: 1.16.3
matplotlib: 3.10.0
seaborn: 0.13.2
scikit-posthocs: 0.13.0


## 4. Dataset Loading and Inspection

This section loads the original binary classification datasets used in the experiments and performs a basic inspection of their structure.

The following datasets are used:

- **Online Shoppers Purchasing Intention Dataset**
- **Bank Marketing Dataset**

After loading the datasets, a brief inspection is performed to verify:

- dataset dimensions
- feature data types
- missing values
- class distributions

In [3]:

# File paths
shoppers_path = os.path.join(
    ORIGINAL_DATASETS_PATH,
    "online_shoppers_intention.csv"
)

bank_path = os.path.join(
    ORIGINAL_DATASETS_PATH,
    "bank_marketing.csv"
)

# Load datasets
shoppers = pd.read_csv(shoppers_path)
bank = pd.read_csv(bank_path)

# Preview
print("Online Shoppers dataset preview:")
display(shoppers.head())

print("\nBank Marketing dataset preview:")
display(bank.head())

Online Shoppers dataset preview:


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False



Bank Marketing dataset preview:


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [4]:
# ==================================================
# Dataset Inspection
# ==================================================

def inspect_dataset(df, target, name):

    print("\n====================================")
    print(f"{name.upper()} DATASET")
    print("====================================")

    print("\nShape:", df.shape)

    print("\nFeature types:")
    print(df.dtypes)

    print("\nMissing values:")
    print(df.isnull().sum())

    print(f"\nTarget distribution ({target}):")
    print(df[target].value_counts())

    print("\nClass proportions:")
    print(df[target].value_counts(normalize=True))


inspect_dataset(shoppers, "Revenue", "Online Shoppers")

inspect_dataset(bank, "y", "Bank Marketing")


ONLINE SHOPPERS DATASET

Shape: (12330, 18)

Feature types:
Administrative               int64
Administrative_Duration    float64
Informational                int64
Informational_Duration     float64
ProductRelated               int64
ProductRelated_Duration    float64
BounceRates                float64
ExitRates                  float64
PageValues                 float64
SpecialDay                 float64
Month                       object
OperatingSystems             int64
Browser                      int64
Region                       int64
TrafficType                  int64
VisitorType                 object
Weekend                       bool
Revenue                       bool
dtype: object

Missing values:
Administrative             0
Administrative_Duration    0
Informational              0
Informational_Duration     0
ProductRelated             0
ProductRelated_Duration    0
BounceRates                0
ExitRates                  0
PageValues                 0
SpecialDay       

## 5. Feature and Target Definition

In this section, the predictor variables (**X**) and target variables (**y**) are defined for both binary datasets.

For the **Online Shoppers dataset**, the target variable is `Revenue`, which indicates whether a session resulted in a purchase. The variables `Weekend` and `Revenue` are converted to integers (`0` / `1`) for consistent numerical representation.

For the **Bank Marketing dataset**, the target variable `y` is converted from categorical values (`yes` / `no`) to binary numerical labels (`1` / `0`).

After defining the feature matrices and target variables, feature types are identified. Categorical variables are detected based on their data types and are later transformed using one-hot encoding, while numerical variables are passed directly to the models.

In [5]:

# Online Shoppers dataset

# Convert boolean → int
shoppers["Weekend"] = shoppers["Weekend"].astype(int)
shoppers["Revenue"] = shoppers["Revenue"].astype(int)

X_shoppers = shoppers.drop(columns=["Revenue"])
y_shoppers = shoppers["Revenue"]


# Bank Marketing dataset


# Convert target from yes/no → 1/0
bank["y"] = bank["y"].map({"yes": 1, "no": 0})

X_bank = bank.drop(columns=["y"])
y_bank = bank["y"]


# Feature type identification
# Online Shoppers

categorical_features_shoppers = X_shoppers.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numerical_features_shoppers = X_shoppers.select_dtypes(
    exclude=["object", "category"]
).columns.tolist()

# Bank Marketing

categorical_features_bank = X_bank.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numerical_features_bank = X_bank.select_dtypes(
    exclude=["object", "category"]
).columns.tolist()

# Verification

print("\nOnline Shoppers feature matrix:", X_shoppers.shape)
print("Online Shoppers target:", y_shoppers.shape)

print("\nBank feature matrix:", X_bank.shape)
print("Bank target:", y_bank.shape)

print("\nOnline Shoppers categorical features:", categorical_features_shoppers)
print("Online Shoppers numerical features:", numerical_features_shoppers)

print("\nBank categorical features:", categorical_features_bank)
print("Bank numerical features:", numerical_features_bank)


Online Shoppers feature matrix: (12330, 17)
Online Shoppers target: (12330,)

Bank feature matrix: (45211, 16)
Bank target: (45211,)

Online Shoppers categorical features: ['Month', 'VisitorType']
Online Shoppers numerical features: ['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'Weekend']

Bank categorical features: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']
Bank numerical features: ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']


## 6. Preprocessing Pipeline

Before training the models, input features must be transformed into a format suitable for machine learning algorithms.

Both binary datasets contain categorical input variables, which are converted into numerical representations using **one-hot encoding**. Numerical variables are passed directly to the model.

Preprocessing is implemented using **scikit-learn’s ColumnTransformer**, which allows different transformations to be applied to different feature groups. Integrating preprocessing into a pipeline ensures that transformations are applied consistently during cross-validation and prevents data leakage.

In [6]:

# Preprocessing Pipeline

# Common categorical transformer
categorical_transformer = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)


# Online Shoppers dataset


preprocessor_shoppers = ColumnTransformer(
    transformers=[
        ("cat", categorical_transformer, categorical_features_shoppers),
        ("num", "passthrough", numerical_features_shoppers)
    ]
)

print("Online Shoppers preprocessing defined.")


# Bank Marketing dataset


preprocessor_bank = ColumnTransformer(
    transformers=[
        ("cat", categorical_transformer, categorical_features_bank),
        ("num", "passthrough", numerical_features_bank)
    ]
)

print("Bank preprocessing defined.")

Online Shoppers preprocessing defined.
Bank preprocessing defined.


## 7. Model Definitions

This section initializes the machine learning algorithms evaluated in the experiments.

At this stage, only model objects are created. The models will later be combined with the preprocessing pipeline and used during hyperparameter optimization and evaluation.

All models are stored in a dictionary, where the key represents the algorithm name and the value is the corresponding model object. This structure allows the experiment to iterate automatically over all algorithms under identical experimental conditions.

In [7]:

# Model Definitions

models = {

    # Decision Tree (CART)
    "DecisionTree": DecisionTreeClassifier(
        random_state=SEED
    ),

    # Random Forest
    "RandomForest": RandomForestClassifier(
        random_state=SEED,
        n_jobs=1
    ),

    # Extra Trees
    "ExtraTrees": ExtraTreesClassifier(
        random_state=SEED,
        n_jobs=1
    ),

    # Gradient Boosting
    "GradientBoosting": GradientBoostingClassifier(
        random_state=SEED
    ),

    # XGBoost
    "XGBoost": xgb.XGBClassifier(
        random_state=SEED,
        n_jobs=1,
        verbosity=0,
        eval_metric="logloss"
    ),

    # LightGBM
    "LightGBM": lgb.LGBMClassifier(
        random_state=SEED,
        n_jobs=1,
        verbosity=-1
    ),

    # CatBoost
    "CatBoost": cb.CatBoostClassifier(
        random_state=SEED,
        verbose=0,
        thread_count=1
    )
}

print("Models initialized:", list(models.keys()))

Models initialized: ['DecisionTree', 'RandomForest', 'ExtraTrees', 'GradientBoosting', 'XGBoost', 'LightGBM', 'CatBoost']


## 8. Hyperparameter Optimization

Hyperparameters are optimized using **Randomized Search with stratified five-fold cross-validation**.

The tuning procedure is performed once for each original binary dataset. If previously tuned hyperparameters are already available in the repository, they are loaded directly in order to avoid repeating the computationally expensive tuning stage.

**F1-score** is used as the optimization metric because it reflects the balance between precision and recall and is consistent with the primary evaluation metric used in the binary experiments.

In [8]:

# Hyperparameter Optimization


import json
import requests
import warnings
warnings.filterwarnings("ignore")

# Hyperparameter search spaces


param_distributions = {

    "DecisionTree": {
        "model__max_depth": [None, 5, 10, 20, 30],
        "model__min_samples_split": [2, 5, 10],
        "model__min_samples_leaf": [1, 2, 4]
    },

    "RandomForest": {
        "model__n_estimators": [100, 200, 300],
        "model__max_depth": [None, 10, 20, 30],
        "model__max_features": ["sqrt", "log2"],
        "model__min_samples_split": [2, 5, 10],
        "model__min_samples_leaf": [1, 2, 4]
    },

    "ExtraTrees": {
        "model__n_estimators": [100, 200, 300],
        "model__max_depth": [None, 10, 20, 30],
        "model__max_features": ["sqrt", "log2"],
        "model__min_samples_split": [2, 5, 10],
        "model__min_samples_leaf": [1, 2, 4]
    },

    "GradientBoosting": {
        "model__n_estimators": [100, 200, 300],
        "model__learning_rate": [0.01, 0.05, 0.1],
        "model__subsample": [0.8, 1.0],
        "model__max_depth": [3, 5, 7],
        "model__max_features": ["sqrt", "log2"],
        "model__min_samples_split": [2, 5]
    },

    "XGBoost": {
        "model__n_estimators": [100, 200, 300],
        "model__learning_rate": [0.01, 0.05, 0.1],
        "model__subsample": [0.8, 1.0],
        "model__colsample_bytree": [0.8, 1.0],
        "model__max_depth": [3, 5, 7]
    },

    "LightGBM": {
        "model__n_estimators": [100, 200, 300],
        "model__learning_rate": [0.01, 0.05, 0.1],
        "model__num_leaves": [31, 50, 70],
        "model__feature_fraction": [0.8, 1.0]
    },

    "CatBoost": {
        "model__iterations": [200, 400, 600],
        "model__learning_rate": [0.01, 0.05, 0.1],
        "model__depth": [4, 6, 8],
        "model__l2_leaf_reg": [1, 3, 5]
    }
}

# Tuning function (F1)


def tune_models(X, y, preprocessor):

    best_params = {}

    for model_name, model in models.items():

        print(f"Tuning {model_name}...")

        pipeline = Pipeline([
            ("preprocessing", preprocessor),
            ("model", model)
        ])

        search = RandomizedSearchCV(
            estimator=pipeline,
            param_distributions=param_distributions[model_name],
            n_iter=10,
            cv=cv,
            scoring="f1",
            n_jobs=1,
            random_state=SEED,
            verbose=0
        )

        search.fit(X, y)

        best_params[model_name] = {
            "best_params": search.best_params_,
            "best_score": search.best_score_
        }

        print("Best score:", round(search.best_score_, 4))

    return best_params



# GitHub URLs


SHOPPERS_URL = "https://raw.githubusercontent.com/Ilaha-Habibova/Tree-algorithms-dataset-characteristics/main/config/best_params_shoppers_f1.json"
BANK_URL = "https://raw.githubusercontent.com/Ilaha-Habibova/Tree-algorithms-dataset-characteristics/main/config/best_params_bank_f1.json"


# Local paths

shoppers_params_path = os.path.join(CONFIG_PATH, "best_params_shoppers_f1.json")
bank_params_path = os.path.join(CONFIG_PATH, "best_params_bank_f1.json")

os.makedirs(CONFIG_PATH, exist_ok=True)


# Load function

def load_from_github(url):
    try:
        r = requests.get(url)
        if r.status_code == 200:
            print(f"🌐 Loaded from GitHub: {url}")
            return r.json()
        else:
            print(f"❌ GitHub file not found: {url}")
            return None
    except Exception as e:
        print(f"⚠️ GitHub load failed: {e}")
        return None


# Load or tune

tuned_now = False

best_params_shoppers = load_from_github(SHOPPERS_URL)
best_params_bank = load_from_github(BANK_URL)



# Fallback: local

if best_params_shoppers is None and os.path.exists(shoppers_params_path):
    print("📂 Loading shoppers params from local...")
    with open(shoppers_params_path) as f:
        best_params_shoppers = json.load(f)

if best_params_bank is None and os.path.exists(bank_params_path):
    print("📂 Loading bank params from local...")
    with open(bank_params_path) as f:
        best_params_bank = json.load(f)

# Run tuning if missing
if best_params_shoppers is None or best_params_bank is None:

    print("\n🚀 Running hyperparameter tuning (F1)...")

    if best_params_shoppers is None:
        best_params_shoppers = tune_models(
            X_shoppers,
            y_shoppers,
            preprocessor_shoppers
        )
        with open(shoppers_params_path, "w") as f:
            json.dump(best_params_shoppers, f, indent=4)
        tuned_now = True

    if best_params_bank is None:
        best_params_bank = tune_models(
            X_bank,
            y_bank,
            preprocessor_bank
        )
        with open(bank_params_path, "w") as f:
            json.dump(best_params_bank, f, indent=4)
        tuned_now = True

    print("\n💾 Binary hyperparameters tuned and saved.")

else:
    print("\n✅ Binary hyperparameters loaded (GitHub/local).")


# Download ONLY if tuning was executed

if tuned_now:
    try:
        from google.colab import files
        print("\n⬇️ Downloading newly tuned hyperparameters...")
        files.download(shoppers_params_path)
        files.download(bank_params_path)
    except:
        print("Download skipped (not in Colab).")
else:
    print("\n📁 No download needed (loaded from GitHub/local).")

🌐 Loaded from GitHub: https://raw.githubusercontent.com/Ilaha-Habibova/Tree-algorithms-dataset-characteristics/main/config/best_params_shoppers_f1.json
🌐 Loaded from GitHub: https://raw.githubusercontent.com/Ilaha-Habibova/Tree-algorithms-dataset-characteristics/main/config/best_params_bank_f1.json

✅ Binary hyperparameters loaded (GitHub/local).

📁 No download needed (loaded from GitHub/local).


## 9. Dataset Size Experiment

This experiment evaluates how classifier performance changes as dataset size increases.

Three dataset sizes are considered:

- Small: 1,000 observations
- Medium: 3,000 observations
- Large: 9,000 observations

Dataset subsets are generated using **nested stratified sampling** in order to preserve class distribution across all size levels.

To improve computational efficiency and reproducibility:

- generated subsets are stored as CSV files
- if the files already exist, they are loaded instead of regenerated

Model evaluation is later performed using stratified five-fold cross-validation with previously optimized hyperparameters.

In [9]:

# DATASET SIZE — BINARY

from sklearn.model_selection import StratifiedShuffleSplit
import os, shutil, requests, zipfile
import numpy as np
import pandas as pd

size_levels = [1000, 3000, 9000]

SIZE_DATASETS_PATH = os.path.join(
    BASE_PATH,
    "datasets",
    "generated_datasets",
    "size_experiments"
)
os.makedirs(SIZE_DATASETS_PATH, exist_ok=True)

# File paths

shoppers_files = {
    s: os.path.join(SIZE_DATASETS_PATH, f"shoppers_size_{s}.csv")
    for s in size_levels
}

bank_files = {
    s: os.path.join(SIZE_DATASETS_PATH, f"bank_size_{s}.csv")
    for s in size_levels
}


# Nested subset function

def create_nested_subsets(X, y, sizes, seed=SEED):

    X = pd.DataFrame(X).reset_index(drop=True)
    y = pd.Series(y).reset_index(drop=True)

    subsets = {}
    selected = np.array([], dtype=int)

    for size in sorted(sizes):

        if len(selected) == 0:
            splitter = StratifiedShuffleSplit(
                n_splits=1,
                train_size=size,
                random_state=seed
            )
            idx, _ = next(splitter.split(X, y))
            selected = idx

        else:
            remaining = np.setdiff1d(np.arange(len(X)), selected)

            X_rem = X.iloc[remaining]
            y_rem = y.iloc[remaining]

            needed = size - len(selected)

            splitter = StratifiedShuffleSplit(
                n_splits=1,
                train_size=needed,
                random_state=seed
            )
            add_idx, _ = next(splitter.split(X_rem, y_rem))
            add_idx = remaining[add_idx]

            selected = np.concatenate([selected, add_idx])

        subsets[size] = (
            X.iloc[selected].copy().reset_index(drop=True),
            y.iloc[selected].copy().reset_index(drop=True)
        )

    return subsets

# GitHub ZIP

ZIP_URL = "https://raw.githubusercontent.com/Ilaha-Habibova/Tree-algorithms-dataset-characteristics/main/datasets/generated_datasets/size_experiments/binary_size_subsets.zip"

zip_path = os.path.join(SIZE_DATASETS_PATH, "binary_size_subsets.zip")

print("🌐 Checking GitHub...")

github_ok = False

try:
    r = requests.get(ZIP_URL)

    if r.status_code == 200:
        with open(zip_path, "wb") as f:
            f.write(r.content)

        shutil.unpack_archive(zip_path, SIZE_DATASETS_PATH)

        github_ok = True
        print("✅ Loaded from GitHub")
        print(f"   Source: {ZIP_URL}")

    else:
        print("❌ GitHub not available")

except:
    print("⚠️ GitHub error")


# Check files

all_files = list(shoppers_files.values()) + list(bank_files.values())
files_exist = all(os.path.exists(f) for f in all_files)

# LOAD

if files_exist:

    shoppers_size_subsets = {}
    bank_size_subsets = {}

    for s in size_levels:

        df = pd.read_csv(shoppers_files[s])
        shoppers_size_subsets[s] = (
            df.drop(columns=["Revenue"]),
            df["Revenue"]
        )

        df = pd.read_csv(bank_files[s])
        bank_size_subsets[s] = (
            df.drop(columns=["y"]),
            df["y"]
        )

    source = "GitHub" if github_ok else "Local"
    print(f"📂 Loaded subsets ({source})")


# GENERATE

else:

    print("🚀 Generating subsets...")

    shoppers_size_subsets = create_nested_subsets(
        X_shoppers, y_shoppers, size_levels, SEED
    )

    bank_size_subsets = create_nested_subsets(
        X_bank, y_bank, size_levels, SEED
    )

    for s in size_levels:

        Xs, ys = shoppers_size_subsets[s]
        pd.concat([Xs, ys], axis=1).to_csv(shoppers_files[s], index=False)

        Xb, yb = bank_size_subsets[s]
        pd.concat([Xb, yb], axis=1).to_csv(bank_files[s], index=False)

    print("📂 Generated locally")

    # ZIP
    zip_file = os.path.join(BASE_PATH, "binary_size_subsets.zip")

    with zipfile.ZipFile(zip_file, 'w') as z:
        for s in size_levels:
            z.write(shoppers_files[s], os.path.basename(shoppers_files[s]))
            z.write(bank_files[s], os.path.basename(bank_files[s]))

    print("📦 ZIP created")

    try:
        from google.colab import files
        files.download(zip_file)
    except:
        pass

🌐 Checking GitHub...
✅ Loaded from GitHub
   Source: https://raw.githubusercontent.com/Ilaha-Habibova/Tree-algorithms-dataset-characteristics/main/datasets/generated_datasets/size_experiments/binary_size_subsets.zip
📂 Loaded subsets (GitHub)


## 10. Dataset Size Experiment – Model Evaluation

This section evaluates model performance on the generated dataset size subsets.

For each dataset size condition, all algorithms are evaluated using stratified five-fold cross-validation. The best hyperparameters obtained during the tuning stage are reused without further optimization.

The primary evaluation metric is **F1-score**.

Additional metrics are also reported:

- precision
- recall
- balanced accuracy
- training time
- model size

The results are stored in tabular form and sorted by F1-score so that the best-performing algorithms can be identified clearly for each dataset size condition.

In [10]:

# DATASET SIZE RESULTS (GITHUB → LOCAL → RUN)

import os
import requests
import pandas as pd
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline
from sklearn.base import clone

RESULTS_PATH = os.path.join(BASE_PATH, "results")
os.makedirs(RESULTS_PATH, exist_ok=True)

size_RESULTS_URL = "https://raw.githubusercontent.com/Ilaha-Habibova/Tree-algorithms-dataset-characteristics/main/results/dataset_size_results_binary.csv"

size_results_path = os.path.join(
    RESULTS_PATH,
    "dataset_size_results_binary.csv"
)

# STEP 1 — GitHub

print("🌐 Checking GitHub for dataset size results...")

results_size_df = None
ran_now = False

try:
    r = requests.get(size_RESULTS_URL)

    if r.status_code == 200:
        with open(size_results_path, "wb") as f:
            f.write(r.content)

        results_size_df = pd.read_csv(size_results_path)

        print(f"✅ Loaded from GitHub")
        print(f"   Source: {size_RESULTS_URL}")

    else:
        print("❌ GitHub not available")

except:
    print("⚠️ GitHub error")


# STEP 2 — Local
if results_size_df is None and os.path.exists(size_results_path):
    print("📂 Loading dataset size results from local...")
    results_size_df = pd.read_csv(size_results_path)


# STEP 3 — Run experiment if missing

if results_size_df is None:

    print("🚀 Running dataset size experiment...")

    results_size = []

    def evaluate_binary(subsets, dataset_name, best_params, preprocessor):

        for size in sorted(subsets.keys()):

            X_subset, y_subset = subsets[size]

            for model_name, base_model in models.items():

                model = clone(base_model)

                if model_name in best_params:
                    tuned_params = {
                        k.replace("model__", ""): v
                        for k, v in best_params[model_name]["best_params"].items()
                    }
                    model.set_params(**tuned_params)

                pipeline = Pipeline([
                    ("preprocessing", preprocessor),
                    ("model", model)
                ])

                scoring = {
                    "balanced_accuracy": "balanced_accuracy",
                    "f1": "f1",
                    "precision": "precision",
                    "recall": "recall"
                }

                cv_results = cross_validate(
                    pipeline,
                    X_subset,
                    y_subset,
                    cv=cv,
                    scoring=scoring,
                    n_jobs=1
                )

                pipeline.fit(X_subset, y_subset)

                results_size.append({
                    "dataset": dataset_name,
                    "size": size,
                    "model": model_name,
                    "balanced_accuracy": cv_results["test_balanced_accuracy"].mean(),
                    "f1": cv_results["test_f1"].mean(),
                    "precision": cv_results["test_precision"].mean(),
                    "recall": cv_results["test_recall"].mean(),
                    "training_time": cv_results["fit_time"].mean(),
                    "model_size_kb": get_model_size_kb(pipeline.named_steps["model"])
                })

    evaluate_binary(
        shoppers_size_subsets,
        "OnlineShoppers",
        best_params_shoppers,
        preprocessor_shoppers
    )

    evaluate_binary(
        bank_size_subsets,
        "Bank",
        best_params_bank,
        preprocessor_bank
    )

    results_size_df = pd.DataFrame(results_size)

    results_size_df.to_csv(size_results_path, index=False)
    ran_now = True

    print("💾 Dataset size results saved")


# STEP 4 — Download if new

if ran_now:
    try:
        from google.colab import files
        print("⬇️ Downloading dataset size results...")
        files.download(size_results_path)
    except:
        print("Download skipped")
else:
    print("📁 No download needed (loaded from GitHub/local)")

🌐 Checking GitHub for dataset size results...
✅ Loaded from GitHub
   Source: https://raw.githubusercontent.com/Ilaha-Habibova/Tree-algorithms-dataset-characteristics/main/results/dataset_size_results_binary.csv
📁 No download needed (loaded from GitHub/local)


## **Dataset Size Experiment – Results visualization**

In [11]:
from IPython.display import display, HTML
import pandas as pd
import uuid

# 1. CONSOLIDATED STYLE AND SCRIPTS
display(HTML("""
<link rel="stylesheet" href="https://cdn.datatables.net/1.13.8/css/jquery.dataTables.min.css">
<script src="https://code.jquery.com/jquery-3.7.1.min.js"></script>
<script src="https://cdn.datatables.net/1.13.8/js/jquery.dataTables.min.js"></script>

<style>
.dataset-block { width: 100%; display: flex; flex-direction: column; align-items: flex-start; margin: 10px 0; }
.dataset-title { text-align: left !important; color: #0b3d91; font-weight: bold; font-size: 16px; margin-bottom: 4px; }
.size-title { text-align: left !important; font-weight: bold; font-size: 13px; margin-bottom: 2px; }

table.dataTable.custom-table {
    margin-left: 0 !important; margin-right: auto !important;
    border-collapse: collapse !important; font-size: 12px !important;
    width: auto !important; table-layout: auto !important; border: 1px solid #888 !important;
}

table.dataTable.custom-table thead th {
    position: relative;
    padding: 4px 28px 4px 10px !important;
    background-color: #dbeaf7 !important;
    text-align: center !important;
    border: 1px solid #bbb !important;
    color: #000000 !important;
    white-space: nowrap;
    background-image: none !important;
}

table.dataTable thead th.sorting:before,
table.dataTable thead th.sorting:after,
table.dataTable thead th.sorting_asc:before,
table.dataTable thead th.sorting_asc:after,
table.dataTable thead th.sorting_desc:before,
table.dataTable thead th.sorting_desc:after {
    display: none !important;
}

table.dataTable.custom-table thead th.sorting {
    background-image: url("data:image/svg+xml,%3Csvg xmlns='http://www.w3.org/2000/svg' width='14' height='14' viewBox='0 0 24 24' fill='none' stroke='black' stroke-width='2' stroke-linecap='round' stroke-linejoin='round'%3E%3Cpath d='M7 15l5 5 5-5M7 9l5-5 5 5'/%3E%3C/svg%3E") !important;
    background-repeat: no-repeat !important;
    background-position: center right 6px !important;
    background-size: 14px 14px !important;
}

table.dataTable.custom-table thead th.sorting_asc {
    background-image: url("data:image/svg+xml,%3Csvg xmlns='http://www.w3.org/2000/svg' width='14' height='14' viewBox='0 0 24 24' fill='none' stroke='black' stroke-width='2.5' stroke-linecap='round' stroke-linejoin='round'%3E%3Cpath d='M18 15l-6-6-6 6'/%3E%3C/svg%3E") !important;
    background-repeat: no-repeat !important;
    background-position: center right 6px !important;
    background-size: 14px 14px !important;
}


table.dataTable.custom-table thead th.sorting_desc {
    background-image: url("data:image/svg+xml,%3Csvg xmlns='http://www.w3.org/2000/svg' width='14' height='14' viewBox='0 0 24 24' fill='none' stroke='black' stroke-width='2.5' stroke-linecap='round' stroke-linejoin='round'%3E%3Cpath d='M6 9l6 6 6-6'/%3E%3C/svg%3E") !important;
    background-repeat: no-repeat !important;
    background-position: center right 6px !important;
    background-size: 14px 14px !important;
}

table.dataTable.custom-table tbody td {
    padding: 2px 8px !important; border: 1px solid #ddd !important;
    text-align: center !important; background-color: #f4f8fc !important;
    color: #000000 !important; line-height: 1.1 !important;
}
table.dataTable.custom-table tbody td:first-child { text-align: left !important; font-weight: bold; }

.dataTables_wrapper .dataTables_filter, .dataTables_wrapper .dataTables_length,
.dataTables_wrapper .dataTables_info, .dataTables_wrapper .dataTables_paginate { display: none !important; }
</style>
"""))

display(HTML("<h3 style='text-align: left; margin-bottom: 10px;'>DATASET SIZE EXPERIMENT RESULTS — BINARY</h3>"))

# 2. DATA PROCESSING AND RENDERING
for dataset in results_size_df["dataset"].unique():
    dataset_df = results_size_df[results_size_df["dataset"] == dataset]

    display(HTML(f"<div class='dataset-block'>"))
    display(HTML(f"<div class='dataset-title'>{dataset.upper()}</div>"))

    for size in sorted(dataset_df["size"].unique()):
        subset = dataset_df[dataset_df["size"] == size]

        # Define columns for Binary
        columns = ["model", "f1", "precision", "recall", "balanced_accuracy", "training_time", "model_size_kb"]

        # Prepare table
        table = subset[columns].round(4).sort_values(by="f1", ascending=False).reset_index(drop=True)
        table_id = f"tbl_{uuid.uuid4().hex[:10]}"
        table_html = table.to_html(index=False, table_id=table_id, classes="display custom-table")

        html_block = f"""
            <div class="size-title">Dataset size = {size}</div>
            {table_html}
            <div style="height:12px;"></div>

            <script>
            (function() {{
                function initTable() {{
                    if (window.jQuery && $.fn.DataTable) {{
                        if (!$.fn.DataTable.isDataTable("#{table_id}")) {{
                            $("#{table_id}").DataTable({{
                                paging: false,
                                searching: false,
                                info: false,
                                order: [[1, "desc"]],
                                autoWidth: false,
                                columnDefs: [{{
                                    targets: "_all",
                                    orderSequence: ["desc", "asc", ""]
                                }}]
                            }});
                        }}
                    }} else {{
                        setTimeout(initTable, 100);
                    }}
                }}
                initTable();
            }})();
            </script>
        """
        display(HTML(html_block))
    display(HTML("</div>"))

model,f1,precision,recall,balanced_accuracy,training_time,model_size_kb
CatBoost,0.6407,0.7100,0.5871,0.7711,1.0092,202.5459
RandomForest,0.6151,0.7539,0.5226,0.7453,0.6211,876.0596
XGBoost,0.6035,0.6456,0.5677,0.7549,0.3308,325.2197
GradientBoosting,0.5887,0.6840,0.5226,0.7382,1.2698,790.4600
DecisionTree,0.5691,0.6158,0.5548,0.7413,0.0216,4.4775
LightGBM,0.5612,0.5924,0.5355,0.7334,0.2761,681.6455
ExtraTrees,0.4685,0.7351,0.3484,0.6624,0.4074,3066.7559


model,f1,precision,recall,balanced_accuracy,training_time,model_size_kb
CatBoost,0.6369,0.7204,0.5733,0.7662,1.1285,202.6943
XGBoost,0.6233,0.6755,0.5797,0.7642,0.1664,326.6123
GradientBoosting,0.6219,0.7076,0.5582,0.7578,0.8138,839.9834
LightGBM,0.6203,0.6953,0.5626,0.7586,0.1946,677.9473
RandomForest,0.6091,0.7556,0.5151,0.7420,0.5867,2397.7666
DecisionTree,0.5910,0.6545,0.5430,0.7459,0.0272,5.4150
ExtraTrees,0.5149,0.7806,0.3902,0.6847,0.2664,7370.0605


model,f1,precision,recall,balanced_accuracy,training_time,model_size_kb
CatBoost,0.6634,0.7239,0.6131,0.7850,1.8223,202.5928
LightGBM,0.6590,0.7225,0.6066,0.7819,0.3336,677.8936
GradientBoosting,0.6573,0.7211,0.6045,0.7807,1.7223,861.9297
XGBoost,0.6572,0.7259,0.6009,0.7796,0.2708,333.6709
RandomForest,0.6462,0.7543,0.5657,0.7659,1.0228,6543.6865
DecisionTree,0.6367,0.6951,0.5987,0.7742,0.0387,6.0400
ExtraTrees,0.5575,0.7970,0.4293,0.7047,0.6801,16967.2744


model,f1,precision,recall,balanced_accuracy,training_time,model_size_kb
XGBoost,0.4588,0.5898,0.3779,0.6725,0.2378,442.5352
LightGBM,0.4491,0.6040,0.3601,0.6648,0.1738,1011.9287
DecisionTree,0.4465,0.5106,0.4033,0.6767,0.0158,7.4463
GradientBoosting,0.4450,0.6408,0.3420,0.6580,0.6664,1254.4580
CatBoost,0.3925,0.5740,0.3000,0.6353,2.3915,1643.1992
RandomForest,0.2947,0.6031,0.1975,0.5914,0.2144,1277.0410
ExtraTrees,0.1656,0.4776,0.1025,0.5445,0.1624,1705.9170


model,f1,precision,recall,balanced_accuracy,training_time,model_size_kb
CatBoost,0.5210,0.6511,0.4358,0.7022,3.0401,1635.8242
XGBoost,0.5147,0.6266,0.4388,0.7018,0.2429,514.9844
GradientBoosting,0.4941,0.6406,0.4045,0.6870,1.3649,1780.3281
LightGBM,0.4867,0.6145,0.4047,0.6853,0.2669,1087.0859
DecisionTree,0.4410,0.5476,0.3703,0.6646,0.0317,15.1025
RandomForest,0.3752,0.6846,0.2593,0.6217,0.4003,3544.4902
ExtraTrees,0.3006,0.7248,0.1937,0.5917,0.4364,4750.9639


model,f1,precision,recall,balanced_accuracy,training_time,model_size_kb
GradientBoosting,0.5054,0.6140,0.4302,0.6972,2.3062,2061.4727
CatBoost,0.5049,0.6259,0.4236,0.6950,3.6349,1638.7773
XGBoost,0.4964,0.6063,0.4207,0.6923,0.4708,604.7656
LightGBM,0.4928,0.5932,0.4226,0.6921,0.4046,1073.8535
DecisionTree,0.4487,0.5760,0.3685,0.6662,0.0743,26.6680
RandomForest,0.4160,0.6470,0.3077,0.6428,0.9897,9696.9609
ExtraTrees,0.3362,0.6620,0.2261,0.6054,0.8817,12870.6133


In [12]:
# VISUALIZATION — CONDITION-WISE RANKING (SIZE)
from IPython.display import display, HTML

MODEL_DISPLAY_NAMES = {
    "DecisionTree": "Decision Tree",
    "RandomForest": "Random Forest",
    "ExtraTrees": "Extra Trees",
    "GradientBoosting": "Gradient Boosting",
    "XGBoost": "XGBoost",
    "LightGBM": "LightGBM",
    "CatBoost": "CatBoost"
}

def pretty_model_name(name):
    return MODEL_DISPLAY_NAMES.get(name, name)

display(HTML("""
<style>
.condition-box {
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    margin: 12px 0;
    padding: 10px 12px;
    background: #ffffff;
    border: 1px solid #d9e3f0;
    border-radius: 8px;
    width: 340px;
    box-shadow: 1px 1px 4px rgba(0,0,0,0.04);
}
.condition-title {
    font-size: 14px;
    font-weight: 700;
    color: #0b3d91;
    border-bottom: 2px solid #7ea6d8;
    margin-bottom: 8px;
    padding-bottom: 4px;
    text-transform: uppercase;
    letter-spacing: 0.2px;
}
.rank-table {
    width: 100%;
    border-collapse: collapse;
}
.rank-row {
    border-bottom: 1px solid #edf2f7;
}
.rank-row:last-child {
    border-bottom: none;
}
.rank-cell {
    padding: 5px 3px;
    font-size: 13px;
    color: #000 !important;
    line-height: 1.15;
}
.rank-icon {
    width: 34px;
    text-align: center;
    font-weight: 700;
}
.rank-model {
    font-weight: 600;
}
.rank-val {
    text-align: right;
    font-family: Consolas, 'Courier New', monospace;
    font-weight: 700;
    width: 56px;
}
</style>
"""))

display(HTML("<h2 style='color:#000; margin: 6px 0 10px 0; font-size: 22px;'>Condition-wise rankings — binary dataset size (F1)</h2>"))

for size in sorted(results_size_df["size"].unique()):
    df_size = results_size_df[results_size_df["size"] == size].copy()
    df_size["rank"] = df_size.groupby("dataset")["f1"].rank(ascending=False, method="min")
    avg_rank = df_size.groupby("model")["rank"].mean().sort_values()

    html_output = f"""
    <div class="condition-box">
        <div class="condition-title">Dataset size = {size}</div>
        <table class="rank-table">
    """

    for i, (model, val) in enumerate(avg_rank.items()):
        icon = {0: "🥇", 1: "🥈", 2: "🥉"}.get(i, f"#{i+1}")
        model_display = pretty_model_name(model)

        html_output += f"""
        <tr class="rank-row">
            <td class="rank-cell rank-icon">{icon}</td>
            <td class="rank-cell rank-model">{model_display}</td>
            <td class="rank-cell rank-val">{val:.2f}</td>
        </tr>
        """

    html_output += "</table></div>"
    display(HTML(html_output))

🥇,XGBoost,2.00
🥈,CatBoost,3.00
🥉,Gradient Boosting,4.00
#4,Decision Tree,4.00
#5,Random Forest,4.00
#6,LightGBM,4.00
#7,Extra Trees,7.00


🥇,CatBoost,1.00
🥈,XGBoost,2.00
🥉,Gradient Boosting,3.00
#4,LightGBM,4.00
#5,Random Forest,5.50
#6,Decision Tree,5.50
#7,Extra Trees,7.00


🥇,CatBoost,1.50
🥈,Gradient Boosting,2.00
🥉,LightGBM,3.00
#4,XGBoost,3.50
#5,Random Forest,5.50
#6,Decision Tree,5.50
#7,Extra Trees,7.00


In [13]:
from IPython.display import display, HTML

MODEL_DISPLAY_NAMES = {
    "DecisionTree": "Decision Tree",
    "RandomForest": "Random Forest",
    "ExtraTrees": "Extra Trees",
    "GradientBoosting": "Gradient Boosting",
    "XGBoost": "XGBoost",
    "LightGBM": "LightGBM",
    "CatBoost": "CatBoost"
}

def pretty_model_name(name):
    return MODEL_DISPLAY_NAMES.get(name, name)

display(HTML("""
<style>
.ranking-container {
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    margin-top: 12px;
    padding: 10px 12px;
    background-color: #ffffff;
    border-radius: 8px;
    border: 1px solid #d9e3f0;
    width: fit-content;
    min-width: 280px;
    box-shadow: 1px 1px 4px rgba(0,0,0,0.04);
}
.ranking-header {
    color: #0b3d91;
    font-size: 14px;
    font-weight: 700;
    border-bottom: 2px solid #7ea6d8;
    margin-bottom: 8px;
    padding-bottom: 4px;
    text-transform: uppercase;
    letter-spacing: 0.2px;
}
.ranking-table {
    width: 100%;
    border-collapse: collapse;
}
.ranking-row {
    border-bottom: 1px solid #edf2f7;
}
.ranking-row:last-child {
    border-bottom: none;
}
.rank-num {
    font-weight: 700;
    font-size: 13px;
    width: 34px;
    text-align: center;
    color: #000000 !important;
    padding: 6px 2px;
    line-height: 1.1;
}
.model-name {
    padding: 6px 6px;
    font-size: 13px;
    font-weight: 600;
    color: #000000 !important;
    line-height: 1.1;
}
.avg-rank-val {
    text-align: right;
    padding: 6px 4px 6px 10px;
    font-family: Consolas, 'Courier New', monospace;
    font-size: 13px;
    font-weight: 700;
    color: #000000 !important;
    min-width: 52px;
    line-height: 1.1;
}
</style>
"""))

display(HTML("<h2 style='color:#000; margin: 6px 0 10px 5px; font-size: 22px;'>Global ranking — binary dataset size experiment</h2>"))

df = results_size_df.copy()
df["rank"] = df.groupby(["dataset", "size"])["f1"].rank(ascending=False, method="min")

global_rank = df.groupby("model")["rank"].mean().sort_values().round(4)

rank_df = global_rank.reset_index()
rank_df.columns = ["model", "avg_rank"]
rank_df["final_rank"] = rank_df["avg_rank"].rank(method="dense").astype(int)

html = """
<div class="ranking-container">
    <div class="ranking-header">Global performance (F1)</div>
    <table class="ranking-table">
"""

for _, row in rank_df.iterrows():
    r = int(row["final_rank"])
    rank_display = {1: "🥇", 2: "🥈", 3: "🥉"}.get(r, f"#{r}")
    model_display = pretty_model_name(row["model"])

    html += f"""
    <tr class="ranking-row">
        <td class="rank-num">{rank_display}</td>
        <td class="model-name">{model_display}</td>
        <td class="avg-rank-val">{row['avg_rank']:.2f}</td>
    </tr>
    """

html += "</table></div>"
display(HTML(html))

🥇,CatBoost,1.83
🥈,XGBoost,2.50
🥉,Gradient Boosting,3.00
#4,LightGBM,3.67
#5,Random Forest,5.00
#5,Decision Tree,5.00
#6,Extra Trees,7.00


## 11. Class Imbalance Manipulation

This section investigates how classification algorithms behave when class distributions become increasingly imbalanced.

The severity of imbalance is measured using the **Imbalance Ratio (IR)**:

\[
IR = \frac{\text{largest class size}}{\text{smallest class size}}
\]

Three imbalance levels are examined:

- IR = 1 — balanced baseline condition  
- IR = 4 — moderate imbalance  
- IR = 9 — severe imbalance  

To ensure that performance differences arise only from changes in class distribution, the dataset size is kept constant in all imbalance experiments:

\[
N = 3600
\]

For binary datasets, class counts are directly controlled according to the selected imbalance ratio while keeping the total number of observations fixed. This procedure is applied to both the Online Shoppers and Bank Marketing datasets.

Each imbalance condition represents a separate dataset version that is later used in model evaluation.

In [14]:

# PART 1 — Class Imbalance Setup

import os
import numpy as np
import pandas as pd

N_IMBALANCE = 3600
imbalance_levels = [1, 4, 9]

IMBALANCE_DATASETS_PATH = os.path.join(
    BASE_PATH,
    "datasets",
    "generated_datasets",
    "imbalance_experiments"
)

os.makedirs(IMBALANCE_DATASETS_PATH, exist_ok=True)

shoppers_files = {
    ir: os.path.join(IMBALANCE_DATASETS_PATH, f"shoppers_IR_{ir}.csv")
    for ir in imbalance_levels
}

bank_files = {
    ir: os.path.join(IMBALANCE_DATASETS_PATH, f"bank_IR_{ir}.csv")
    for ir in imbalance_levels
}

def detect_binary_labels(y):
    counts = pd.Series(y).value_counts()
    return counts.idxmax(), counts.idxmin()

shoppers_major, shoppers_minor = detect_binary_labels(y_shoppers)
bank_major, bank_minor = detect_binary_labels(y_bank)

print("Online Shoppers class distribution:")
print(pd.Series(y_shoppers).value_counts())

print("\nBank class distribution:")
print(pd.Series(y_bank).value_counts())

def generate_binary_counts(N, R):
    minority = int(round(N / (R + 1)))
    majority = N - minority
    return majority, minority

def sample_by_class_counts(X, y, class_counts, seed=SEED):

    X = pd.DataFrame(X)
    y = pd.Series(y)

    rng = np.random.RandomState(seed)
    sampled = []

    for cls, count in class_counts.items():
        idx = y[y == cls].index
        chosen = rng.choice(idx, size=count, replace=False)
        sampled.extend(chosen)

    sampled = np.array(sampled)

    return (
        X.loc[sampled].reset_index(drop=True),
        y.loc[sampled].reset_index(drop=True)
    )

Online Shoppers class distribution:
Revenue
0    10422
1     1908
Name: count, dtype: int64

Bank class distribution:
y
0    39922
1     5289
Name: count, dtype: int64


In [15]:
# CLASS IMBALANCE (FULL PIPELINE)

import requests
import zipfile

ZIP_URL = "https://raw.githubusercontent.com/Ilaha-Habibova/Tree-algorithms-dataset-characteristics/main/datasets/generated_datasets/imbalance_experiments/binary_imbalance_subsets.zip"

zip_path = os.path.join(IMBALANCE_DATASETS_PATH, "binary_imbalance_subsets.zip")


# STEP 1 — GitHub
print("🌐 Checking GitHub...")

github_ok = False

try:
    r = requests.get(ZIP_URL)

    if r.status_code == 200:
        with open(zip_path, "wb") as f:
            f.write(r.content)

        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(IMBALANCE_DATASETS_PATH)

        github_ok = True

        print("🌐 Loaded from GitHub")
        print(f"   Source: {ZIP_URL}")

    else:
        print("❌ GitHub not available")

except:
    print("⚠️ GitHub error")

# STEP 2 — CHECK FILES

all_files = list(shoppers_files.values()) + list(bank_files.values())
files_exist = all(os.path.exists(f) for f in all_files)

# STEP 3 — LOAD

if files_exist:

    source = "GitHub" if github_ok else "Local"
    print(f"📂 Loading ({source})")

    shoppers_imbalance_subsets = {}
    bank_imbalance_subsets = {}

    for ir in imbalance_levels:

        df = pd.read_csv(shoppers_files[ir])
        shoppers_imbalance_subsets[ir] = (
            df.drop(columns=["Revenue"]),
            df["Revenue"]
        )

        df = pd.read_csv(bank_files[ir])
        bank_imbalance_subsets[ir] = (
            df.drop(columns=["y"]),
            df["y"]
        )

    print("📁 No download needed")

# STEP 4 — GENERATE (fallback)

else:

    print("🚀 Generating binary imbalance datasets...")

    shoppers_imbalance_subsets = {}
    bank_imbalance_subsets = {}

    for ir in imbalance_levels:

        # ONLINE SHOPPERS
        maj, mino = generate_binary_counts(N_IMBALANCE, ir)

        counts = {
            shoppers_major: maj,
            shoppers_minor: mino
        }

        Xs, ys = sample_by_class_counts(X_shoppers, y_shoppers, counts)
        shoppers_imbalance_subsets[ir] = (Xs, ys)

        pd.concat([Xs, ys], axis=1).to_csv(
            shoppers_files[ir],
            index=False
        )

        # BANK DATASET
        maj, mino = generate_binary_counts(N_IMBALANCE, ir)

        counts = {
            bank_major: maj,
            bank_minor: mino
        }

        Xb, yb = sample_by_class_counts(X_bank, y_bank, counts)
        bank_imbalance_subsets[ir] = (Xb, yb)

        pd.concat([Xb, yb], axis=1).to_csv(
            bank_files[ir],
            index=False
        )

    # ZIP
    zip_file = os.path.join(BASE_PATH, "binary_imbalance_subsets.zip")

    with zipfile.ZipFile(zip_file, 'w') as z:
        for ir in imbalance_levels:
            z.write(shoppers_files[ir], os.path.basename(shoppers_files[ir]))
            z.write(bank_files[ir], os.path.basename(bank_files[ir]))

    print("📦 Clean ZIP created")

    try:
        from google.colab import files
        files.download(zip_file)
    except:
        pass

🌐 Checking GitHub...
🌐 Loaded from GitHub
   Source: https://raw.githubusercontent.com/Ilaha-Habibova/Tree-algorithms-dataset-characteristics/main/datasets/generated_datasets/imbalance_experiments/binary_imbalance_subsets.zip
📂 Loading (GitHub)
📁 No download needed


## 13. Class Imbalance Experiment – Model Evaluation

This section evaluates the performance of the seven machine learning algorithms under different class imbalance conditions.

Each imbalance level is treated as a separate dataset version.

The same evaluation procedure used in the dataset size experiment is applied here:

- stratified five-fold cross-validation  
- F1-score as the primary evaluation metric  
- precision, recall, and balanced accuracy as complementary metrics  
- measurement of training time and model size

The tuned hyperparameters obtained earlier are reused so that algorithm comparisons remain fair and consistent across all generated dataset versions.

In [16]:
# CLASS IMBALANCE RESULTS (GITHUB → LOCAL → RUN)

import os
import requests
import pandas as pd
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline
from sklearn.base import clone

RESULTS_PATH = os.path.join(BASE_PATH, "results")
os.makedirs(RESULTS_PATH, exist_ok=True)

IMBALANCE_RESULTS_URL = "https://raw.githubusercontent.com/Ilaha-Habibova/Tree-algorithms-dataset-characteristics/main/results/imbalance_results_binary.csv"

imbalance_results_path = os.path.join(
    RESULTS_PATH,
    "imbalance_results_binary.csv"
)

print("🌐 Checking GitHub for binary imbalance results...")

results_imbalance_df = None
ran_now = False

try:
    r = requests.get(IMBALANCE_RESULTS_URL)

    if r.status_code == 200:
        with open(imbalance_results_path, "wb") as f:
            f.write(r.content)

        results_imbalance_df = pd.read_csv(imbalance_results_path)

        print(f"✅ Loaded from GitHub: {IMBALANCE_RESULTS_URL}")

    else:
        print("❌ GitHub not available")

except:
    print("⚠️ GitHub error")


if results_imbalance_df is None and os.path.exists(imbalance_results_path):
    print("📂 Loading binary results from local...")
    results_imbalance_df = pd.read_csv(imbalance_results_path)


if results_imbalance_df is None:

    print("🚀 Running binary imbalance experiment...")

    results_imbalance = []

    def evaluate_binary(subsets, dataset_name, best_params, preprocessor):

        for IR in sorted(subsets.keys()):

            X_subset, y_subset = subsets[IR]

            for model_name, base_model in models.items():

                model = clone(base_model)

                if model_name in best_params:
                    tuned_params = {
                        k.replace("model__", ""): v
                        for k, v in best_params[model_name]["best_params"].items()
                    }
                    model.set_params(**tuned_params)

                pipeline = Pipeline([
                    ("preprocessing", preprocessor),
                    ("model", model)
                ])

                scoring = {
                    "balanced_accuracy": "balanced_accuracy",
                    "f1": "f1",
                    "precision": "precision",
                    "recall": "recall"
                }

                cv_results = cross_validate(
                    pipeline,
                    X_subset,
                    y_subset,
                    cv=cv,
                    scoring=scoring,
                    n_jobs=1
                )

                pipeline.fit(X_subset, y_subset)

                results_imbalance.append({
                    "dataset": dataset_name,
                    "imbalance_ratio": IR,
                    "model": model_name,
                    "balanced_accuracy": cv_results["test_balanced_accuracy"].mean(),
                    "f1": cv_results["test_f1"].mean(),
                    "precision": cv_results["test_precision"].mean(),
                    "recall": cv_results["test_recall"].mean(),
                    "training_time": cv_results["fit_time"].mean(),
                    "model_size_kb": get_model_size_kb(pipeline.named_steps["model"])
                })

    evaluate_binary(
        shoppers_imbalance_subsets,
        "OnlineShoppers",
        best_params_shoppers,
        preprocessor_shoppers
    )

    evaluate_binary(
        bank_imbalance_subsets,
        "Bank",
        best_params_bank,
        preprocessor_bank
    )

    results_imbalance_df = pd.DataFrame(results_imbalance)

    results_imbalance_df.to_csv(imbalance_results_path, index=False)
    ran_now = True

    print("💾 Binary imbalance results saved")


if ran_now:
    try:
        from google.colab import files
        print("⬇️ Downloading binary results...")
        files.download(imbalance_results_path)
    except:
        print("Download skipped")
else:
    print("📁 No download needed (loaded from GitHub/local)")

🌐 Checking GitHub for binary imbalance results...
✅ Loaded from GitHub: https://raw.githubusercontent.com/Ilaha-Habibova/Tree-algorithms-dataset-characteristics/main/results/imbalance_results_binary.csv
📁 No download needed (loaded from GitHub/local)


## **Class Imbalance Experiment – Results visualization**

In [17]:
from IPython.display import display, HTML
import pandas as pd
import uuid

display(HTML("""
<link rel="stylesheet" href="https://cdn.datatables.net/1.13.8/css/jquery.dataTables.min.css">
<script src="https://code.jquery.com/jquery-3.7.1.min.js"></script>
<script src="https://cdn.datatables.net/1.13.8/js/jquery.dataTables.min.js"></script>

<style>
.dataset-block { width: 100%; display: flex; flex-direction: column; align-items: flex-start; margin: 10px 0; }
.dataset-title { text-align: left !important; color: #0b3d91; font-weight: bold; font-size: 16px; margin-bottom: 4px; }
.size-title { text-align: left !important; font-weight: bold; font-size: 13px; margin-bottom: 2px; }

table.dataTable.custom-table {
    margin-left: 0 !important; margin-right: auto !important;
    border-collapse: collapse !important; font-size: 12px !important;
    width: auto !important; table-layout: auto !important;
    border: 1px solid #888 !important;
}

table.dataTable.custom-table thead th,
table.dataTable.custom-table thead th.sorting,
table.dataTable.custom-table thead th.sorting_asc,
table.dataTable.custom-table thead th.sorting_desc {
    position: relative;
    padding: 4px 28px 4px 10px !important;
    background-color: #dbeaf7 !important;
    text-align: center !important;
    border: 1px solid #bbb !important;
    background-image: none !important;
    color: #000000 !important;
}

table.dataTable thead th.sorting:before,
table.dataTable thead th.sorting:after,
table.dataTable thead th.sorting_asc:before,
table.dataTable thead th.sorting_asc:after,
table.dataTable thead th.sorting_desc:before,
table.dataTable thead th.sorting_desc:after {
    display: none !important;
}

table.dataTable.custom-table tbody td {
    padding: 2px 8px !important;
    border: 1px solid #ddd !important;
    text-align: center !important;
    background-color: #f4f8fc !important;
    color: #000000 !important;
}

table.dataTable.custom-table tbody td:first-child {
    text-align: left !important;
    font-weight: bold;
}

table.dataTable.custom-table thead th.sorting {
    background-image: url("data:image/svg+xml,%3Csvg xmlns='http://www.w3.org/2000/svg' width='14' height='14' viewBox='0 0 24 24' fill='none' stroke='black' stroke-width='2'%3E%3Cpath d='M7 15l5 5 5-5M7 9l5-5 5 5'/%3E%3C/svg%3E") !important;
    background-repeat: no-repeat !important;
    background-position: center right 6px !important;
}

table.dataTable.custom-table thead th.sorting_asc {
    background-image: url("data:image/svg+xml,%3Csvg xmlns='http://www.w3.org/2000/svg' width='14' height='14' viewBox='0 0 24 24' fill='none' stroke='black' stroke-width='2.5'%3E%3Cpath d='M18 15l-6-6-6 6'/%3E%3C/svg%3E") !important;
}

table.dataTable.custom-table thead th.sorting_desc {
    background-image: url("data:image/svg+xml,%3Csvg xmlns='http://www.w3.org/2000/svg' width='14' height='14' viewBox='0 0 24 24' fill='none' stroke='black' stroke-width='2.5'%3E%3Cpath d='M6 9l6 6 6-6'/%3E%3C/svg%3E") !important;
}

.dataTables_wrapper .dataTables_filter,
.dataTables_wrapper .dataTables_length,
.dataTables_wrapper .dataTables_info,
.dataTables_wrapper .dataTables_paginate {
    display: none !important;
}
</style>
"""))

display(HTML("<h3 style='text-align: left;'>CLASS IMBALANCE RESULTS — BINARY</h3>"))

for dataset in results_imbalance_df["dataset"].unique():

    dataset_df = results_imbalance_df[results_imbalance_df["dataset"] == dataset]

    display(HTML("<div class='dataset-block'>"))
    display(HTML(f"<div class='dataset-title'>{dataset.upper()}</div>"))

    for IR in sorted(dataset_df["imbalance_ratio"].unique()):

        subset = dataset_df[dataset_df["imbalance_ratio"] == IR]

        columns = [
            "model",
            "f1",
            "precision",
            "recall",
            "balanced_accuracy",
            "training_time",
            "model_size_kb"
        ]

        table = subset[columns].round(4)\
            .sort_values(by="f1", ascending=False)\
            .reset_index(drop=True)

        table_id = f"tbl_{uuid.uuid4().hex[:10]}"
        table_html = table.to_html(index=False, table_id=table_id, classes="display custom-table")

        html_block = f"""
        <div class="size-title">Imbalance Ratio = {IR}</div>
        {table_html}
        <div style="height:12px;"></div>

        <script>
        (function() {{
            function initTable() {{
                if (window.jQuery && $.fn.DataTable) {{
                    if (!$.fn.DataTable.isDataTable("#{table_id}")) {{
                        $("#{table_id}").DataTable({{
                            paging: false,
                            searching: false,
                            info: false,
                            order: [[1, "desc"]],
                            autoWidth: false,
                            columnDefs: [{{
                                targets: "_all",
                                orderSequence: ["desc", "asc", ""]
                            }}]
                        }});
                    }}
                }} else {{
                    setTimeout(initTable, 100);
                }}
            }}
            initTable();
        }})();
        </script>
        """

        display(HTML(html_block))

    display(HTML("</div>"))

model,f1,precision,recall,balanced_accuracy,training_time,model_size_kb
RandomForest,0.8560,0.8565,0.8556,0.8561,1.1198,3952.3027
CatBoost,0.8531,0.8692,0.8378,0.8558,1.0721,202.3271
XGBoost,0.8504,0.8526,0.8483,0.8508,0.3034,330.3965
GradientBoosting,0.8503,0.8564,0.8444,0.8514,0.9447,839.6934
LightGBM,0.8499,0.8451,0.8550,0.8492,0.2867,677.4668
DecisionTree,0.8477,0.8530,0.8428,0.8486,0.0243,6.0400
ExtraTrees,0.8381,0.8170,0.8606,0.8339,0.3483,11841.3369


model,f1,precision,recall,balanced_accuracy,training_time,model_size_kb
CatBoost,0.6841,0.7343,0.6431,0.7920,1.2417,202.7334
LightGBM,0.6645,0.7264,0.6153,0.7781,0.2078,677.8066
DecisionTree,0.6599,0.7209,0.6097,0.7752,0.0212,6.0400
XGBoost,0.6599,0.7197,0.6125,0.7757,0.1952,325.1475
GradientBoosting,0.6598,0.7238,0.6083,0.7747,0.9075,842.6523
RandomForest,0.6593,0.7455,0.5944,0.7714,0.6015,3184.9062
ExtraTrees,0.5914,0.7868,0.4764,0.7215,0.4466,9497.7432


model,f1,precision,recall,balanced_accuracy,training_time,model_size_kb
CatBoost,0.5493,0.6590,0.4722,0.7225,1.1593,202.8584
LightGBM,0.5487,0.6482,0.4778,0.7244,0.2123,679.4258
GradientBoosting,0.5454,0.6791,0.4583,0.7170,1.0840,841.5303
XGBoost,0.5384,0.6227,0.4750,0.7215,0.1734,327.6807
RandomForest,0.5083,0.7577,0.3833,0.6847,0.4481,2340.1016
DecisionTree,0.5042,0.6479,0.4278,0.6998,0.0201,5.8838
ExtraTrees,0.3913,0.7613,0.2639,0.6273,0.3050,7259.6006


model,f1,precision,recall,balanced_accuracy,training_time,model_size_kb
CatBoost,0.8666,0.8425,0.8928,0.8625,3.0267,1646.5586
GradientBoosting,0.8631,0.8400,0.8878,0.8592,1.6271,2066.5352
XGBoost,0.8586,0.8380,0.8806,0.8550,0.2774,577.5996
RandomForest,0.8565,0.8282,0.8872,0.8514,0.5214,7307.4297
LightGBM,0.8559,0.8371,0.8756,0.8525,0.2829,1074.7656
ExtraTrees,0.8244,0.8318,0.8178,0.8258,0.3689,8246.5508
DecisionTree,0.7956,0.7975,0.7944,0.7961,0.0344,25.8867


model,f1,precision,recall,balanced_accuracy,training_time,model_size_kb
CatBoost,0.6464,0.7048,0.5986,0.7677,3.1565,1611.5273
XGBoost,0.6451,0.6953,0.6028,0.7682,0.2760,548.4473
LightGBM,0.6385,0.6803,0.6042,0.7663,0.2920,1078.1152
GradientBoosting,0.6368,0.6977,0.5875,0.7616,1.5346,1817.7344
RandomForest,0.5758,0.6987,0.4917,0.7191,0.5131,5631.8008
DecisionTree,0.5638,0.6242,0.5194,0.7205,0.0344,19.3213
ExtraTrees,0.4859,0.6845,0.3778,0.6670,0.3694,6887.2793


model,f1,precision,recall,balanced_accuracy,training_time,model_size_kb
CatBoost,0.4515,0.6040,0.3667,0.6693,2.9974,1610.4414
XGBoost,0.4243,0.5707,0.3389,0.6549,0.2686,523.7441
LightGBM,0.4057,0.5457,0.3278,0.6483,0.2890,1085.1348
GradientBoosting,0.4052,0.5696,0.3167,0.6448,1.5047,1744.0381
DecisionTree,0.3752,0.4536,0.3250,0.6401,0.0343,15.7275
RandomForest,0.2763,0.6101,0.1833,0.5844,0.4803,3951.3652
ExtraTrees,0.1889,0.5249,0.1167,0.5528,0.3628,5218.5342


In [18]:
# VISUALIZATION — CONDITION-WISE RANKING (IMBALANCE)
from IPython.display import display, HTML

MODEL_DISPLAY_NAMES = {
    "DecisionTree": "Decision Tree",
    "RandomForest": "Random Forest",
    "ExtraTrees": "Extra Trees",
    "GradientBoosting": "Gradient Boosting",
    "XGBoost": "XGBoost",
    "LightGBM": "LightGBM",
    "CatBoost": "CatBoost"
}

def pretty_model_name(name):
    return MODEL_DISPLAY_NAMES.get(name, name)

style_html = """
<style>
    .condition-box {
        font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
        margin: 12px 0;
        padding: 10px 12px;
        background: #ffffff;
        border: 1px solid #d9e3f0;
        border-radius: 8px;
        width: 340px;
        box-shadow: 1px 1px 4px rgba(0,0,0,0.04);
    }
    .condition-title {
        font-size: 14px;
        font-weight: 700;
        color: #0b3d91;
        border-bottom: 2px solid #7ea6d8;
        margin-bottom: 8px;
        padding-bottom: 4px;
        text-transform: uppercase;
        letter-spacing: 0.2px;
    }
    .rank-table {
        width: 100%;
        border-collapse: collapse;
    }
    .rank-row {
        border-bottom: 1px solid #edf2f7;
    }
    .rank-row:last-child {
        border-bottom: none;
    }
    .rank-cell {
        padding: 5px 3px;
        font-size: 13px;
        color: #000 !important;
        line-height: 1.15;
    }
    .rank-icon {
        width: 34px;
        text-align: center;
        font-weight: 700;
    }
    .rank-model {
        font-weight: 600;
    }
    .rank-val {
        text-align: right;
        font-family: Consolas, 'Courier New', monospace;
        font-weight: 700;
        width: 56px;
    }
</style>
"""
display(HTML(style_html))
display(HTML("<h2 style='color:#000; margin: 6px 0 10px 0; font-size: 22px;'>📊 Imbalance-specific rankings — binary (F1)</h2>"))

for IR in sorted(results_imbalance_df["imbalance_ratio"].unique()):
    df_imb = results_imbalance_df[
        results_imbalance_df["imbalance_ratio"] == IR
    ].copy()

    df_imb["rank"] = df_imb.groupby("dataset")["f1"].rank(ascending=False, method="min")
    avg_rank = df_imb.groupby("model")["rank"].mean().sort_values()

    html_output = f"""
    <div class="condition-box">
        <div class="condition-title">Imbalance ratio = {IR}</div>
        <table class="rank-table">
    """

    for i, (model, val) in enumerate(avg_rank.items()):
        icon = {0: "🥇", 1: "🥈", 2: "🥉"}.get(i, f"#{i+1}")
        model_display = pretty_model_name(model)

        html_output += f"""
        <tr class="rank-row">
            <td class="rank-cell rank-icon">{icon}</td>
            <td class="rank-cell rank-model">{model_display}</td>
            <td class="rank-cell rank-val">{val:.2f}</td>
        </tr>
        """

    html_output += "</table></div>"
    display(HTML(html_output))

🥇,CatBoost,1.50
🥈,Random Forest,2.50
🥉,XGBoost,3.00
#4,Gradient Boosting,3.00
#5,LightGBM,5.00
#6,Decision Tree,6.50
#7,Extra Trees,6.50


🥇,CatBoost,1.00
🥈,LightGBM,2.50
🥉,XGBoost,2.50
#4,Gradient Boosting,4.50
#5,Decision Tree,5.00
#6,Random Forest,5.50
#7,Extra Trees,7.00


🥇,CatBoost,1.00
🥈,LightGBM,2.50
🥉,XGBoost,3.00
#4,Gradient Boosting,3.50
#5,Random Forest,5.50
#6,Decision Tree,5.50
#7,Extra Trees,7.00


In [19]:
from IPython.display import display, HTML
import pandas as pd

MODEL_DISPLAY_NAMES = {
    "DecisionTree": "Decision Tree",
    "RandomForest": "Random Forest",
    "ExtraTrees": "Extra Trees",
    "GradientBoosting": "Gradient Boosting",
    "XGBoost": "XGBoost",
    "LightGBM": "LightGBM",
    "CatBoost": "CatBoost"
}

def pretty_model_name(name):
    return MODEL_DISPLAY_NAMES.get(name, name)

style_html = """
<style>
    .ranking-container {
        font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
        margin-top: 12px;
        padding: 10px 12px;
        background-color: #ffffff;
        border-radius: 8px;
        border: 1px solid #d9e3f0;
        width: fit-content;
        min-width: 280px;
        box-shadow: 1px 1px 4px rgba(0,0,0,0.04);
    }
    .ranking-header {
        color: #0b3d91;
        font-size: 14px;
        font-weight: 700;
        border-bottom: 2px solid #7ea6d8;
        margin-bottom: 8px;
        padding-bottom: 4px;
        text-transform: uppercase;
        letter-spacing: 0.2px;
    }
    .ranking-table {
        width: 100%;
        border-collapse: collapse;
    }
    .ranking-row {
        border-bottom: 1px solid #edf2f7;
    }
    .ranking-row:last-child {
        border-bottom: none;
    }
    .rank-num {
        font-weight: 700;
        font-size: 13px;
        width: 34px;
        text-align: center;
        color: #000000 !important;
        padding: 6px 2px;
        line-height: 1.1;
    }
    .model-name {
        padding: 6px 6px;
        font-size: 13px;
        font-weight: 600;
        color: #000000 !important;
        line-height: 1.1;
    }
    .avg-rank-val {
        text-align: right;
        padding: 6px 4px 6px 10px;
        font-family: Consolas, 'Courier New', monospace;
        font-size: 13px;
        font-weight: 700;
        color: #000000 !important;
        min-width: 52px;
        line-height: 1.1;
    }
</style>
"""

display(HTML(style_html))
display(HTML("<h2 style='color:#000; margin: 6px 0 10px 5px; font-size: 22px;'>🏆 Global ranking — binary (all datasets)</h2>"))

# GLOBAL RANKING LOGIC

df = results_imbalance_df.copy()

# Rank within each dataset + imbalance level
df["rank"] = (
    df.groupby(["dataset", "imbalance_ratio"])["f1"]
    .rank(ascending=False, method="min")
)

# Average rank across ALL datasets + imbalance levels
global_rank = (
    df.groupby("model")["rank"]
    .mean()
    .sort_values()
    .round(4)
)

rank_df = global_rank.reset_index()
rank_df.columns = ["model", "avg_rank"]

# Dense ranking
rank_df["final_rank"] = (
    rank_df["avg_rank"]
    .rank(method="dense")
    .astype(int)
)

html = """
<div class="ranking-container">
    <div class="ranking-header">
        Global performance (F1)
    </div>
    <table class="ranking-table">
"""

for _, row in rank_df.iterrows():

    r = int(row["final_rank"])

    rank_display = {
        1: "🥇",
        2: "🥈",
        3: "🥉"
    }.get(r, f"#{r}")

    model_display = pretty_model_name(row["model"])

    html += f"""
    <tr class="ranking-row">
        <td class="rank-num">{rank_display}</td>
        <td class="model-name">{model_display}</td>
        <td class="avg-rank-val">{row['avg_rank']:.2f}</td>
    </tr>
    """

html += """
    </table>
</div>
"""

display(HTML(html))

🥇,CatBoost,1.17
🥈,XGBoost,2.83
🥉,LightGBM,3.33
#4,Gradient Boosting,3.67
#5,Random Forest,4.50
#6,Decision Tree,5.67
#7,Extra Trees,6.83


## 14. Statistical Significance Testing

To determine whether the observed differences between algorithms are statistically significant, non-parametric statistical tests are applied.

Following common practice in machine learning research, the **Friedman test** is used to evaluate whether significant differences exist among the algorithms across multiple experimental conditions.

The Friedman test compares the **average ranks** of the algorithms rather than their raw performance values. This approach is appropriate when multiple algorithms are evaluated across several datasets or experimental settings.


If the Friedman test rejects the null hypothesis, it indicates that at least one algorithm performs significantly differently from the others.

To identify which specific algorithms differ significantly, the **Nemenyi post-hoc test** is applied. This test compares all pairs of algorithms using their average ranks. If the difference between the average ranks of two algorithms exceeds the **Critical Difference (CD)**, their performance difference is considered statistically significant.

In this binary notebook, statistical analysis is performed using **F1-score**, ensuring consistency with the primary evaluation metric used throughout the experiments.

In [20]:

# CORE FUNCTION — FRIEDMAN + NEMENYI (SHARED)

import numpy as np
from scipy import stats
from scipy.stats import studentized_range

def compute_friedman_nemenyi(df, condition_col, metric):

    pivot = df.pivot_table(
        index=condition_col,
        columns="model",
        values=metric,
        aggfunc="mean"
    )

    pivot = pivot.dropna()

    # Friedman test
    stat, p = stats.friedmanchisquare(
        *[pivot[col].values for col in pivot.columns]
    )

    # Ranking
    ranks = pivot.rank(axis=1, ascending=False)
    avg_ranks = ranks.mean().sort_values()

    # Nemenyi CD
    k = len(pivot.columns)
    N = len(pivot)

    q_alpha = studentized_range.ppf(1 - 0.05, k, np.inf) / np.sqrt(2)
    CD = q_alpha * np.sqrt(k * (k + 1) / (6 * N))

    # Significant pairs
    significant_pairs = []
    models = avg_ranks.index.tolist()

    for i in range(len(models)):
        for j in range(i + 1, len(models)):
            diff = abs(avg_ranks[models[i]] - avg_ranks[models[j]])
            if diff > CD:
                significant_pairs.append((models[i], models[j], diff))

    return stat, p, avg_ranks, CD, significant_pairs

In [21]:
# STATISTICAL SIGNIFICANCE TEST — SIZE


size_df = results_size_df.copy()

# Combine dataset + size → condition
size_df["condition"] = (
    size_df["dataset"] + "_size_" + size_df["size"].astype(str)
)

print("\n" + "="*70)
print("FRIEDMAN TEST — SIZE (BINARY)")
print("="*70)

stat, p, avg_ranks, CD, pairs = compute_friedman_nemenyi(
    size_df,
    "condition",
    "f1"
)

significance = "SIGNIFICANT" if p < 0.05 else "NOT SIGNIFICANT"

print(f"\nFriedman: stat={stat:.4f}, p={p:.6f} → {significance}")

print("\n" + "="*70)
print("NEMENYI POST-HOC — SIZE (BINARY)")
print("="*70)

if p >= 0.05:
    print("No post-hoc analysis (Friedman not significant)")
else:

    print("\nAverage ranks:")
    for m, r in avg_ranks.items():
        print(f"{m:<18} {r:.4f}")

    print(f"\nCritical Difference (CD) = {CD:.4f}")

    if len(pairs) == 0:
        print("No significant pairwise differences")
    else:
        print("\nSignificant pairs:")
        for m1, m2, diff in pairs:
            print(f"{m1} vs {m2} (diff={diff:.4f})")


FRIEDMAN TEST — SIZE (BINARY)

Friedman: stat=24.5000, p=0.000422 → SIGNIFICANT

NEMENYI POST-HOC — SIZE (BINARY)

Average ranks:
CatBoost           1.8333
XGBoost            2.5000
GradientBoosting   3.0000
LightGBM           3.6667
RandomForest       5.0000
DecisionTree       5.0000
ExtraTrees         7.0000

Critical Difference (CD) = 3.6772

Significant pairs:
CatBoost vs ExtraTrees (diff=5.1667)
XGBoost vs ExtraTrees (diff=4.5000)
GradientBoosting vs ExtraTrees (diff=4.0000)


In [22]:

# STATISTICAL SIGNIFICANCE TEST — IMBALANCE

imb_df = results_imbalance_df.copy()

# Combine dataset + IR → condition
imb_df["condition"] = (
    imb_df["dataset"] + "_IR_" + imb_df["imbalance_ratio"].astype(str)
)

print("\n" + "="*70)
print("FRIEDMAN TEST — IMBALANCE (BINARY)")
print("="*70)

stat, p, avg_ranks, CD, pairs = compute_friedman_nemenyi(
    imb_df,
    "condition",
    "f1"
)

significance = "SIGNIFICANT" if p < 0.05 else "NOT SIGNIFICANT"

print(f"\nFriedman: stat={stat:.4f}, p={p:.6f} → {significance}")

print("\n" + "="*70)
print("NEMENYI POST-HOC — IMBALANCE (BINARY)")
print("="*70)

if p >= 0.05:
    print("No post-hoc analysis (Friedman not significant)")
else:

    print("\nAverage ranks:")
    for m, r in avg_ranks.items():
        print(f"{m:<18} {r:.4f}")

    print(f"\nCritical Difference (CD) = {CD:.4f}")

    if len(pairs) == 0:
        print("No significant pairwise differences")
    else:
        print("\nSignificant pairs:")
        for m1, m2, diff in pairs:
            print(f"{m1} vs {m2} (diff={diff:.4f})")


FRIEDMAN TEST — IMBALANCE (BINARY)

Friedman: stat=27.0000, p=0.000145 → SIGNIFICANT

NEMENYI POST-HOC — IMBALANCE (BINARY)

Average ranks:
CatBoost           1.1667
XGBoost            2.8333
LightGBM           3.3333
GradientBoosting   3.6667
RandomForest       4.5000
DecisionTree       5.6667
ExtraTrees         6.8333

Critical Difference (CD) = 3.6772

Significant pairs:
CatBoost vs DecisionTree (diff=4.5000)
CatBoost vs ExtraTrees (diff=5.6667)
XGBoost vs ExtraTrees (diff=4.0000)
